# Calculate effective coverage by quintile and scenario

Also by age, sex, and pregnancy status (though coverage will not vary by pregnancy status due to a lack of data).

This is similar to what the pregnancy simulation does at the individual level, but using groups instead.
It can be shared between multiplication models that do not incorporate individual heterogeneity.

In [1]:
import pandas as pd

In [2]:
location = "nigeria"
vehicle = "rice"
fortificant = "iron"

In [3]:
# Parameters
location = "india"
fortificant = "iron"
vehicle = "rice"


In [4]:
results_dir = f"../results/{fortificant}/{vehicle}"

In [5]:
full_coverage_probability = pd.read_csv(
    f"{results_dir}/baseline_fortification/full_coverage/{location}.csv"
)
full_coverage_probability = full_coverage_probability.set_index(
    [c for c in full_coverage_probability.columns if c != "value"]
).value
full_coverage_probability

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.096246
                            2                rice            0.124174
                            3                rice            0.111760
                            4                rice            0.099788
                            5                rice            0.072762
        5          15       1                rice            0.100522
                            2                rice            0.123632
                            3                rice            0.110857
                            4                rice            0.100010
                            5                rice            0.070413
        15         30       1                rice            0.101619
                            2                rice            0.133421
                            3                rice            0.114570
                            4   

In [6]:
any_coverage_probability = pd.read_csv(
    f"{results_dir}/baseline_fortification/any_coverage/{location}.csv"
)
any_coverage_probability = any_coverage_probability.set_index(
    [c for c in any_coverage_probability.columns if c != "value"]
).value
any_coverage_probability

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.554010
                            2                rice            0.518359
                            3                rice            0.492228
                            4                rice            0.496830
                            5                rice            0.307495
        5          15       1                rice            0.650621
                            2                rice            0.611305
                            3                rice            0.573440
                            4                rice            0.547655
                            5                rice            0.326922
        15         30       1                rice            0.589372
                            2                rice            0.539883
                            3                rice            0.512091
                            4   

In [7]:
partial_coverage_mean = pd.read_csv(
    f"{results_dir}/baseline_fortification/partial_coverage_amount/mean/{location}.csv"
)
partial_coverage_mean = partial_coverage_mean.set_index(
    [c for c in partial_coverage_mean.columns if c != "value"]
).value
partial_coverage_mean

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.529705
                            2                rice            0.539072
                            3                rice            0.534043
                            4                rice            0.504590
                            5                rice            0.472097
        5          15       1                rice            0.561247
                            2                rice            0.559742
                            3                rice            0.553740
                            4                rice            0.530998
                            5                rice            0.483357
        15         30       1                rice            0.532851
                            2                rice            0.549381
                            3                rice            0.550596
                            4   

In [8]:
current_coverage = (
    full_coverage_probability
    + (any_coverage_probability - full_coverage_probability) * partial_coverage_mean
)
current_coverage

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.338726
                            2                rice            0.336668
                            3                rice            0.314946
                            4                rice            0.300132
                            5                rice            0.183579
        5          15       1                rice            0.409264
                            2                rice            0.396603
                            3                rice            0.367008
                            4                rice            0.337709
                            5                rice            0.194398
        15         30       1                rice            0.361518
                            2                rice            0.356724
                            3                rice            0.333444
                            4   

In [9]:
from lsff_utils import config_utils

scenarios = config_utils.get_intervention_scenarios(location, vehicle)
scenarios

['intervention']

In [10]:
any_consumption = pd.read_csv(
    f"../results/{vehicle}/vehicle_consumption/any/{location}.csv"
)
any_consumption = any_consumption.set_index(
    [c for c in any_consumption.columns if c != "value"]
).value
any_consumption

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.890088
                            2                rice            0.875116
                            3                rice            0.873100
                            4                rice            0.884775
                            5                rice            0.865564
        5          15       1                rice            0.961146
                            2                rice            0.980478
                            3                rice            0.989830
                            4                rice            0.992594
                            5                rice            0.988565
        15         30       1                rice            0.976640
                            2                rice            0.970350
                            3                rice            0.988181
                            4   

In [11]:
if (
    len(
        any_consumption.reset_index()[["sex", "age_start", "age_end"]].drop_duplicates()
    )
    == 1
):
    # Assumed same for all ages/sexes
    any_consumption = any_consumption.droplevel(["sex", "age_start", "age_end"])
    display(any_consumption)

In [12]:
fortifiability = pd.read_csv(
    f"../results/{vehicle}/vehicle_consumption/fortifiability/{location}.csv"
)
fortifiability = fortifiability.set_index(
    [c for c in fortifiability.columns if c != "value"]
).value
fortifiability

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.542477
                            2                rice            0.553830
                            3                rice            0.537182
                            4                rice            0.520193
                            5                rice            0.423345
        5          15       1                rice            0.626483
                            2                rice            0.626911
                            3                rice            0.604934
                            4                rice            0.567622
                            5                rice            0.441991
        15         30       1                rice            0.552578
                            2                rice            0.567719
                            3                rice            0.560005
                            4   

In [13]:
import pathlib, numpy as np

for scenario in scenarios:
    intervention_coverage = pd.read_csv(
        f"{results_dir}/{scenario}/intervention_fortification/any_coverage/{location}.csv"
    )
    intervention_coverage = intervention_coverage.set_index(
        [c for c in intervention_coverage.columns if c != "value"]
    ).value
    target_coverage = intervention_coverage * fortifiability
    display(target_coverage)
    assert (
        (target_coverage > current_coverage.reindex_like(target_coverage))
        | np.isclose(target_coverage, current_coverage.reindex_like(target_coverage))
    ).all()
    target_coverage[
        np.isclose(target_coverage, current_coverage.reindex_like(target_coverage))
    ] = current_coverage.reindex_like(target_coverage)
    # Not all coverage is effective -- this is as a proportion of coverage!
    effectiveness = pd.read_csv(
        f"{results_dir}/{scenario}/intervention_fortification/effectiveness/{location}.csv"
    )
    effectiveness = effectiveness.set_index(
        [c for c in effectiveness.columns if c != "value"]
    ).value
    effective_intervention_coverage = any_consumption * target_coverage * effectiveness
    path = f"{results_dir}/{scenario}/intervention_fortification/effective_coverage/{location}.csv"
    pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
    effective_intervention_coverage.reset_index().to_csv(path, index=False)

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.499079
                            2                rice            0.509524
                            3                rice            0.494207
                            4                rice            0.478578
                            5                rice            0.389477
        5          15       1                rice            0.576364
                            2                rice            0.576758
                            3                rice            0.556539
                            4                rice            0.522212
                            5                rice            0.406632
        15         30       1                rice            0.508372
                            2                rice            0.522301
                            3                rice            0.515204
                            4   

In [14]:
# Not all coverage is effective -- this is as a proportion of coverage!
effectiveness = pd.read_csv(
    f"{results_dir}/baseline_fortification/effectiveness/{location}.csv"
)
effectiveness = effectiveness.set_index(
    [c for c in effectiveness.columns if c != "value"]
).value

In [15]:
effective_baseline_coverage = any_consumption * current_coverage * effectiveness
effective_baseline_coverage

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.214062
        5          15       1                rice            0.279287
        15         30       1                rice            0.250682
        30         50       1                rice            0.267612
        50         125      1                rice            0.277209
Male    0          5        1                rice            0.212924
        5          15       1                rice            0.283057
        15         30       1                rice            0.262358
        30         50       1                rice            0.261352
        50         125      1                rice            0.275177
Female  0          5        2                rice            0.209182
        5          15       2                rice            0.276091
        15         30       2                rice            0.245764
        30         50       2   

In [16]:
path = f"{results_dir}/baseline_fortification/effective_coverage/{location}.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
effective_baseline_coverage.reset_index().to_csv(path, index=False)